# Recommendation DataFrame Inspector
This notebook allows for the inspection and comparison of `recommendation_df` outputs generated by the `RecommendationEngine` without recomputing the pipeline. It retrieves data directly from the parquet cache.

## 1. Environment Setup
We establish the project root and update `sys.path` to ensure we can import the `vqs` submodule and the `configs` folder.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd

# Start from the current working directory
cwd = Path.cwd()
PROJECT_ROOT: Path | None = None

# Search upwards (max 5 levels) for the project root
for _ in range(5):
    if (cwd / "dependencies").is_dir() and (cwd / "vqs").is_dir():
        PROJECT_ROOT = cwd
        break
    cwd = cwd.parent

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root. Run Jupyter from within your project folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print(f"Project Root: {PROJECT_ROOT}")
print(f"CWD set to: {os.getcwd()}")

## 2. Load Configuration and Cache
We load the `base_constants.py` file. 

**Note:** If you want to inspect a specific run, ensure the parameters in the cell below match those used during that run.

In [ ]:
from main import load_config
from vqs.cache_management import CacheManager

# Load default config
config_path = PROJECT_ROOT / "configs" / "full_pipeline" / "pipeline_e5_ZH_mini.py" #type: ignore
config = load_config(config_path)

# Replicate RecommendationEngine parameter list for hashing
important_params_list = [
    "data_year", "dist", "alpha", "crw_paper_choice", "rec_dist_method",
    "n_recommendations", "subset_n", "filter_districts"
]
if config.filter_districts:
    important_params_list.append("district")
if getattr(config, 'E5_instruction', None) is not None:
    important_params_list.append("E5_instruction")

# Construct prefix exactly as evaluate_pipeline does
prefix = f"recs_{config.data_year}_{config.dist}_a{config.alpha}_subset={config.subset_n}"
if config.filter_districts:
    prefix += f"_{config.district}"

cacher = CacheManager(
    config=config,
    cache_dir=config.RECOMMENDATION_CACHE_DIR,
    prefix=prefix,
    params_list=important_params_list
)

recommendation_df = cacher.load_if_exists()

if recommendation_df is not None:
    print("✅ Cache successfully loaded.")
else:
    print("❌ No cache found. Check your parameters or prefix logic.")

## 3. Inspection of Headers and Structure
Here we look at the column organization. The dataframe should contain both baseline columns and CRW-prefixed columns.

In [ ]:
if recommendation_df is not None:
    print(f"DataFrame Shape: {recommendation_df.shape}")

    rec_cols = [c for c in recommendation_df.columns]

    print(f"\n--- Recommendation Columns ({len(rec_cols)}) ---")
    for col in rec_cols:
        print(col)


In [ ]:
if recommendation_df is not None:
    x: int = 1 # 1 <= x <= n_recommendations
    print(f"---- Top {x} match IDs (first 20 rows) ---")
    print(recommendation_df[f"_matchID_{x}_L2_sv"].iloc[:20])
    print(recommendation_df[f"CRW__matchID_{x}_L2_sv"].iloc[:20])

## 4. Compare Different Outputs (Extension)
To compare different engine runs (e.g., different Alpha values), you can manually point to specific files in the cache directory.

In [ ]:
import glob

# List all cached recommendation files to see what's available
cache_files = glob.glob(str(config.RECOMMENDATION_CACHE_DIR / "*.parquet"))
print("Available cache files:")
for f in cache_files:
    print(Path(f).name)

def load_specific_cache(filename):
    path = config.RECOMMENDATION_CACHE_DIR / filename
    return pd.read_parquet(path)

# Example: Compare top 1 matches between two runs
# df_v1 = load_specific_cache("some_file_a0.6.parquet")
# df_v2 = load_specific_cache("some_file_a0.8.parquet")